In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib as mpl
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import json

mpl.rcParams.update({
    'axes.titlesize'   : 0,        # no title
    'axes.labelsize'   : 14,       # x and y axis labels
    'xtick.labelsize'  : 12,       # x tick labels
    'ytick.labelsize'  : 12,       # y tick labels
    'legend.fontsize'  : 12,
    'font.size'        : 12,      
})

In [40]:
df = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/main_data.csv", encoding="utf-8")

/tmp/ipykernel_4691/2256054893.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/main_data.csv", encoding="utf-8")


In [41]:
# ============================================================
# Lexicon counting pipeline (single words + multiword phrases)
# for MULTIPLE vocabularies, in a Jupyter notebook
# ============================================================

import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

TEXT_COL = "text"   # change if needed

# ------------------------------------------------------------------
# 0) CONFIG: add all your vocab files here (name -> csv path)
# ------------------------------------------------------------------
LEXICONS = {
    "scapegoating": "scapegoating_vocabulary.csv",
    "scapegoating_them": "scapegoating_them.csv",
    "scapegoating_implicit": "scapegoating_implicit.csv",
    "victimism_us": "victimism_us.csv",
    "victimism_me": "victimism_me.csv",
    "victimism_general": "victimism_general.csv",
}

# If you have a stable id column you like to inspect later:
ID_COLS_TO_SHOW = ["speech_par_id", "name_date"]  # adjust or set []


# ------------------------------------------------------------------
# 1) Helper: load + repair a lexicon file robustly into a DataFrame
# ------------------------------------------------------------------
def load_lexicon_df(path: str) -> pd.DataFrame:
    """
    Robustly load a lexicon CSV that might be:
    - well-formed with a 'term' column (optionally 'category')
    - single-column where each row contains "term,category"
    - single-column with just terms

    Returns a DataFrame with at least 'term' column.
    """
    lex_raw = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

    # Strip BOM/whitespace from column names
    lex_raw.columns = [c.strip().replace("\ufeff", "") for c in lex_raw.columns]

    if lex_raw.shape[1] == 1:
        only_col = lex_raw.columns[0]
        s = lex_raw[only_col].astype(str)

        # Split only on first comma; if there is no comma, category will be NaN
        tmp = s.str.split(",", n=1, expand=True)
        if tmp.shape[1] == 1:
            tmp[1] = np.nan

        tmp.columns = ["term", "category"]

        tmp["term"] = (
            tmp["term"].astype(str)
            .str.replace('"', "", regex=False)
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
        )
        tmp["category"] = (
            tmp["category"].astype(str)
            .str.replace('"', "", regex=False)
            .str.strip()
        )

        # Drop header-like / empty rows
        tmp = tmp[tmp["term"].str.lower().ne("term") & tmp["term"].ne("")]

         # Keep only non-empty terms
        tmp = tmp[tmp["term"].notna() & tmp["term"].ne("")]

         # If category is literally "nan" string, normalize it
        tmp.loc[tmp["category"].str.lower().eq("nan"), "category"] = np.nan

        lex = tmp.copy()

    else:
        # Normal CSV
        if "term" not in lex_raw.columns:
            # fallback: first column as term
            lex_raw = lex_raw.rename(columns={lex_raw.columns[0]: "term"})
        lex = lex_raw.copy()

        lex["term"] = (
            lex["term"].astype(str)
            .str.replace('"', "", regex=False)
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
        )
        lex = lex[lex["term"].str.lower().ne("term") & lex["term"].ne("")]

    # Lowercase for matching, but keep original in case you want it later
    lex["term"] = lex["term"].astype(str).str.lower().str.strip()
    lex = lex.dropna(subset=["term"])

    return lex


# ------------------------------------------------------------------
# 2) Helper: build regex objects for single words + multi-word phrases
# ------------------------------------------------------------------
def build_lexicon_regex(terms: list[str]):
    """
    Returns (single_word_regex, multi_word_regex)
    - single_word_regex uses word boundaries
    - multi_word_regex matches phrases anywhere (escaped)
    """
    terms = [t.strip().lower() for t in terms if isinstance(t, str) and t.strip()]
    terms = list(pd.unique(terms))

    single_words = [t for t in terms if " " not in t]
    multi_words  = [t for t in terms if " " in t]

    single_words_escaped = [re.escape(w) for w in single_words]
    multi_words_escaped  = [re.escape(p) for p in multi_words]

    single_word_regex = (
        re.compile(r"\b(" + "|".join(single_words_escaped) + r")\b", flags=re.IGNORECASE)
        if single_words_escaped else None
    )

    # Sorting multi-words by length helps reduce some partial-match surprises
    multi_words_escaped_sorted = sorted(multi_words_escaped, key=len, reverse=True)
    multi_word_regex = (
        re.compile(r"(" + "|".join(multi_words_escaped_sorted) + r")", flags=re.IGNORECASE)
        if multi_words_escaped_sorted else None
    )

    return single_word_regex, multi_word_regex, len(single_words), len(multi_words)


# ------------------------------------------------------------------
# 3) Helper: counting functions for a given regex pair
# ------------------------------------------------------------------
def make_count_fns(single_word_regex, multi_word_regex):
    def count_total(text):
        if not isinstance(text, str) or not text:
            return 0
        single_hits = single_word_regex.findall(text) if single_word_regex else []
        multi_hits  = multi_word_regex.findall(text) if multi_word_regex else []
        return len(single_hits) + len(multi_hits)

    def count_distinct(text):
        if not isinstance(text, str) or not text:
            return 0
        single_hits = set(m.lower() for m in single_word_regex.findall(text)) if single_word_regex else set()
        multi_hits  = set(m.lower() for m in multi_word_regex.findall(text)) if multi_word_regex else set()
        return len(single_hits.union(multi_hits))

    return count_total, count_distinct


# ------------------------------------------------------------------
# 4) Word counts (for normalization)
# ------------------------------------------------------------------
if "word_count" not in df.columns:
    df["word_count"] = df[TEXT_COL].fillna("").astype(str).str.split().str.len()


# ------------------------------------------------------------------
# 5) Run for each lexicon + write columns
# ------------------------------------------------------------------
lexicon_debug = {}

for lex_name, lex_path in LEXICONS.items():
    print(f"\n=== Processing lexicon: {lex_name} ({lex_path}) ===")

    lex_df = load_lexicon_df(lex_path)
    terms = lex_df["term"].dropna().astype(str).str.lower().str.strip().unique().tolist()

    single_re, multi_re, n_single, n_multi = build_lexicon_regex(terms)
    count_total, count_distinct = make_count_fns(single_re, multi_re)

    print(f"Loaded terms: {len(terms)} | Single: {n_single} | Multi: {n_multi}")

    # Store debug info if you want to inspect later
    lexicon_debug[lex_name] = {
        "path": lex_path,
        "n_terms": len(terms),
        "n_single": n_single,
        "n_multi": n_multi,
        "preview_terms": terms[:10],
    }

    # Column names
    col_count = f"{lex_name}_count"
    col_dist  = f"{lex_name}_distinct"
    col_per100 = f"{lex_name}_per_100w"

    # Apply with progress bar
    df[col_count] = df[TEXT_COL].progress_apply(count_total)
    df[col_dist]  = df[TEXT_COL].progress_apply(count_distinct)

    # Normalize per 100 words (safe for 0 word_count)
    df[col_per100] = np.where(
        df["word_count"] > 0,
        df[col_count] / df["word_count"] * 100,
        0.0
    )

    # Quick sanity check
    print(df[[col_count, col_dist, col_per100]].describe().round(3))


# ------------------------------------------------------------------
# 6) Optional: inspect top hits for each lexicon
# ------------------------------------------------------------------
def show_top_hits(lex_name: str, n: int = 20):
    col_count = f"{lex_name}_count"
    cols = [c for c in ID_COLS_TO_SHOW if c in df.columns] + [TEXT_COL, col_count, f"{lex_name}_per_100w"]
    return df.sort_values(col_count, ascending=False)[cols].head(n)

# Example usage:
# show_top_hits("scapegoating_them", 20)
# show_top_hits("victimism_general", 20)

# ------------------------------------------------------------------
# 7) Save updated dataframe (optional)
# ------------------------------------------------------------------
# df.to_csv("prof_llm_scapegoating_with_lexicon_counts.csv", index=False)



=== Processing lexicon: scapegoating (scapegoating_vocabulary.csv) ===
Loaded terms: 20 | Single: 1 | Multi: 19


  0%|          | 0/71808 [00:00<?, ?it/s]

  0%|          | 0/71808 [00:00<?, ?it/s]

       scapegoating_count  scapegoating_distinct  scapegoating_per_100w
count           71808.000              71808.000              71808.000
mean                0.045                  0.040                  0.050
std                 0.252                  0.205                  0.314
min                 0.000                  0.000                  0.000
25%                 0.000                  0.000                  0.000
50%                 0.000                  0.000                  0.000
75%                 0.000                  0.000                  0.000
max                 6.000                  4.000                 21.053

=== Processing lexicon: scapegoating_them (scapegoating_them.csv) ===
Loaded terms: 60 | Single: 0 | Multi: 60


  0%|          | 0/71808 [00:00<?, ?it/s]

...

  0%|          | 0/71808 [00:00<?, ?it/s]

       victimism_me_count  victimism_me_distinct  victimism_me_per_100w
count           71808.000              71808.000              71808.000
mean                0.000                  0.000                  0.000
std                 0.018                  0.018                  0.019
min                 0.000                  0.000                  0.000
25%                 0.000                  0.000                  0.000
50%                 0.000                  0.000                  0.000
75%                 0.000                  0.000                  0.000
max                 1.000                  1.000                  2.273

=== Processing lexicon: victimism_general (victimism_general.csv) ===
Loaded terms: 51 | Single: 0 | Multi: 51


  0%|          | 0/71808 [00:00<?, ?it/s]

  0%|          | 0/71808 [00:00<?, ?it/s]

       victimism_general_count  victimism_general_distinct  \
count                71808.000                   71808.000   
mean                     0.001                       0.001   
std                      0.034                       0.032   
min                      0.000                       0.000   
25%                      0.000                       0.000   
50%                      0.000                       0.000   
75%                      0.000                       0.000   
max                      2.000                       1.000   

       victimism_general_per_100w  
count                   71808.000  
mean                        0.001  
std                         0.044  
min                         0.000  
25%                         0.000  
50%                         0.000  
75%                         0.000  
max                         5.263  


In [6]:
df["election_year"] = df["election_date"].astype(str).str[:4].astype("Int64")



In [7]:
set(df["election_year"])

{1952,
 1956,
 1960,
 1964,
 1968,
 1972,
 1976,
 1980,
 1984,
 1988,
 1992,
 1996,
 2000,
 2004,
 2008,
 2012,
 2016,
 2020}

In [8]:
df["person"] = (
    df["person"].astype(str)
    + "_"
    + df["election_year"].astype(str)
)


In [9]:
df_pop = df[df["Populism"] == 1]
df_non = df[df["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)

Populist rows: 1861


In [10]:
df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())


1    1861
0    1861
Name: Populism, dtype: int64


In [11]:
df_balanced[["scapegoating_per_100w", "Populism"]].groupby("Populism").describe()


scapegoating_per_100w                                          \
                         count      mean       std  min  25%  50%  75%   
Populism                                                                 
0                       1861.0  0.049843  0.289022  0.0  0.0  0.0  0.0   
1                       1861.0  0.047169  0.299952  0.0  0.0  0.0  0.0   

                    
               max  
Populism            
0         3.797468  
1         5.263158

In [12]:
import statsmodels.api as sm

y = df_balanced["scapegoating_per_100w"]
X = df_balanced[["Populism"]]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())


                              OLS Regression Results                             
Dep. Variable:     scapegoating_per_100w   R-squared:                       0.000
Model:                               OLS   Adj. R-squared:                 -0.000
Method:                    Least Squares   F-statistic:                   0.07665
Date:                   Sat, 17 Jan 2026   Prob (F-statistic):              0.782
Time:                           11:11:09   Log-Likelihood:                -730.71
No. Observations:                   3722   AIC:                             1465.
Df Residuals:                       3720   BIC:                             1478.
Df Model:                              1                                         
Covariance Type:                     HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0498 

In [13]:
import numpy as np
coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["scapegoating_per_100w"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit(cov_type="HC3")
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)


(-0.003828564549517791, 0.006595415781429123)

In [14]:
import statsmodels.formula.api as smf

# Make sure types are correct
df_balanced["Populism"] = df_balanced["Populism"].astype(int)
df_balanced["prior_president"] = df_balanced["prior_president"].astype(int)

# Drop missing values in required columns
df_mixed = df_balanced.dropna(
    subset=["scapegoating_per_100w", "Populism", "prior_president", "person"]
)


In [15]:
mixed = smf.mixedlm(
    formula="scapegoating_per_100w ~ Populism + prior_president",
    data=df_mixed,
    groups=df_mixed["person"]
)

mixed_res = mixed.fit(method="lbfgs")
print(mixed_res.summary())


                 Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  scapegoating_per_100w
No. Observations:   3722     Method:              REML                 
No. Groups:         34       Scale:               0.0858               
Min. group size:    17       Log-Likelihood:      inf                  
Max. group size:    367      Converged:           Yes                  
Mean group size:    109.5                                              
-----------------------------------------------------------------------
                Coef.   Std.Err.    z    P>|z|    [0.025       0.975]  
-----------------------------------------------------------------------
Intercept        0.032 316964.665  0.000 1.000  -621239.295  621239.360
Populism        -0.019      0.012 -1.575 0.115       -0.043       0.005
prior_president  0.004 637813.507  0.000 1.000 -1250091.499 1250091.506
Group Var        0.000                                                 



/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2055: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2246: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  w

In [16]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

fe = smf.ols(
    "scapegoating_per_100w ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe.summary())


                              OLS Regression Results                             
Dep. Variable:     scapegoating_per_100w   R-squared:                       0.012
Model:                               OLS   Adj. R-squared:                  0.002
Method:                    Least Squares   F-statistic:                     19.35
Date:                   Sat, 17 Jan 2026   Prob (F-statistic):           0.000107
Time:                           11:11:10   Log-Likelihood:                -709.11
No. Observations:                   3722   AIC:                             1488.
Df Residuals:                       3687   BIC:                             1706.
Df Model:                             34                                         
Covariance Type:                 cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


In [17]:
df.to_csv("prof_llm_scapegoating.csv", encoding="utf-8", index=False)

In [18]:
dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_scapegoating_with_similarity.csv", encoding="utf-8")

/tmp/ipykernel_4691/2124667189.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_scapegoating_with_similarity.csv", encoding="utf-8")


In [19]:
df_pop = dfllm[dfllm["Populism"] == 1]
df_non = dfllm[dfllm["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)

Populist rows: 1861


In [20]:
df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())

1    1861
0    1861
Name: Populism, dtype: int64


In [21]:
df_balanced[["scapegoating_sim_top10", "scapegoating_sim_max"]].describe()
df_balanced[["scapegoating_sim_top10", "scapegoating_sim_max"]].corr()


,scapegoating_sim_top10,scapegoating_sim_max
scapegoating_sim_top10,1.000000,0.824905
scapegoating_sim_max,0.824905,1.000000


In [22]:
for var in ["scapegoating_sim_top10", "scapegoating_sim_max"]:
    res = smf.ols(
        f"{var} ~ Populism + prior_president + C(person)",
        data=df_balanced
    ).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})
    print(var, res.params["Populism"], res.pvalues["Populism"])


scapegoating_sim_top10 -0.004459726053408797 0.0961651408081681
scapegoating_sim_max 0.004292826611228077 0.250519194914092


In [23]:
import statsmodels.api as sm

y = df_balanced["scapegoating_sim_top10"]
X = sm.add_constant(df_balanced[["Populism"]])

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())

                              OLS Regression Results                              
Dep. Variable:     scapegoating_sim_top10   R-squared:                       0.000
Model:                                OLS   Adj. R-squared:                  0.000
Method:                     Least Squares   F-statistic:                     1.081
Date:                    Sat, 17 Jan 2026   Prob (F-statistic):              0.299
Time:                            11:11:13   Log-Likelihood:                 6440.9
No. Observations:                    3722   AIC:                        -1.288e+04
Df Residuals:                        3720   BIC:                        -1.287e+04
Df Model:                               1                                         
Covariance Type:                      HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       

In [24]:
import numpy as np

coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["scapegoating_sim_top10"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit()
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)

(0.0014809708895186125, 0.0010118691523932119)

In [25]:
import statsmodels.api as sm

y = df_balanced["scapegoating_sim_top10"]

X = df_balanced[["Populism", "prior_president"]]
X = sm.add_constant(X)

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())


                              OLS Regression Results                              
Dep. Variable:     scapegoating_sim_top10   R-squared:                       0.001
Model:                                OLS   Adj. R-squared:                  0.000
Method:                     Least Squares   F-statistic:                     1.138
Date:                    Sat, 17 Jan 2026   Prob (F-statistic):              0.321
Time:                            11:11:14   Log-Likelihood:                 6441.5
No. Observations:                    3722   AIC:                        -1.288e+04
Df Residuals:                        3719   BIC:                        -1.286e+04
Df Model:                               2                                         
Covariance Type:                      HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
co

In [26]:
import statsmodels.formula.api as smf

fe_person = smf.ols(
    "scapegoating_sim_top10 ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe_person.summary())

                              OLS Regression Results                              
Dep. Variable:     scapegoating_sim_top10   R-squared:                       0.082
Model:                                OLS   Adj. R-squared:                  0.074
Method:                     Least Squares   F-statistic:                     1.928
Date:                    Sat, 17 Jan 2026   Prob (F-statistic):              0.174
Time:                            11:11:14   Log-Likelihood:                 6599.6
No. Observations:                    3722   AIC:                        -1.313e+04
Df Residuals:                        3687   BIC:                        -1.291e+04
Df Model:                              34                                         
Covariance Type:                  cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


In [27]:
dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_scapegoating_with_similarity_multi.csv", encoding="utf-8")

/tmp/ipykernel_4691/1457127885.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_scapegoating_with_similarity_multi.csv", encoding="utf-8")


In [28]:
df_pop = dfllm[dfllm["Populism"] == 1]
df_non = dfllm[dfllm["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)

Populist rows: 1861


In [29]:
df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())

1    1861
0    1861
Name: Populism, dtype: int64


In [30]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# -------------------------------------------------------------------
# CONFIG: which dependent variables to run the exact same pipeline on
# -------------------------------------------------------------------
DVS = [
    "victimism_us_sim_top10",
    "victimism_me_sim_top10",
    "scapegoating_them_sim_top10",
]

# Optional: your downsampling pieces (only used in the seed loop).
# Assumes you already have these from before:
# df_pop, df_non, n_pop
DO_SEED_LOOP = True
N_SEEDS = 100


def run_all_models(dv: str, df_balanced):
    print("\n" + "=" * 80)
    print(f"DV = {dv}")
    print("=" * 80)

    # ----------------------------
    # 1) OLS: DV ~ Populism
    # ----------------------------
    y = df_balanced[dv]
    X = sm.add_constant(df_balanced[["Populism"]])
    ols_hc3 = sm.OLS(y, X).fit(cov_type="HC3")
    print("\n[1] OLS (HC3):", f"{dv} ~ Populism")
    print(ols_hc3.summary())

    # ----------------------------
    # 2) Seed loop: downsample non-populists repeatedly, store Populism coef
    # ----------------------------
    if DO_SEED_LOOP:
        coefs = []
        for seed in range(N_SEEDS):
            df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
            df_bal = pd.concat([df_pop, df_non_sampled], ignore_index=True)

            y_seed = df_bal[dv]
            X_seed = sm.add_constant(df_bal[["Populism"]])
            res_seed = sm.OLS(y_seed, X_seed).fit()
            coefs.append(res_seed.params["Populism"])

        coefs = np.array(coefs, dtype=float)
        print(f"\n[2] Seed loop (N={N_SEEDS}) Populism coef:")
        print("    mean =", coefs.mean())
        print("    std  =", coefs.std(ddof=1))
        print("    min  =", coefs.min())
        print("    max  =", coefs.max())

    # ----------------------------
    # 3) OLS: DV ~ Populism + prior_president
    # ----------------------------
    X2 = sm.add_constant(df_balanced[["Populism", "prior_president"]])
    ols2_hc3 = sm.OLS(y, X2).fit(cov_type="HC3")
    print("\n[3] OLS (HC3):", f"{dv} ~ Populism + prior_president")
    print(ols2_hc3.summary())

    # ----------------------------
    # 4) Person FE: DV ~ Populism + prior_president + C(person)
    #    Cluster SE by person (same as your example)
    # ----------------------------
    fe_person = smf.ols(
        f"{dv} ~ Populism + prior_president + C(person)",
        data=df_balanced
    ).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

    print("\n[4] Person FE (clustered by person):",
          f"{dv} ~ Populism + prior_president + C(person)")
    print(fe_person.summary())

    # Return fitted objects if you want to store them
    return {
        "ols_hc3": ols_hc3,
        "ols2_hc3": ols2_hc3,
        "fe_person": fe_person,
        # seed coefs not returned unless you want them
    }


# -------------------------------------------------------------------
# RUN
# -------------------------------------------------------------------
results = {}
for dv in DVS:
    results[dv] = run_all_models(dv, df_balanced)



DV = victimism_us_sim_top10

[1] OLS (HC3): victimism_us_sim_top10 ~ Populism
                              OLS Regression Results                              
Dep. Variable:     victimism_us_sim_top10   R-squared:                       0.001
Model:                                OLS   Adj. R-squared:                  0.001
Method:                     Least Squares   F-statistic:                     3.235
Date:                    Sat, 17 Jan 2026   Prob (F-statistic):             0.0722
Time:                            11:19:42   Log-Likelihood:                 5282.8
No. Observations:                    3722   AIC:                        -1.056e+04
Df Residuals:                        3720   BIC:                        -1.055e+04
Df Model:                               1                                         
Covariance Type:                      HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '



[2] Seed loop (N=100) Populism coef:
    mean = 0.00850188002578326
    std  = 0.0013438189723617606
    min  = 0.005396395658608966
    max  = 0.011292396868609464

[3] OLS (HC3): victimism_me_sim_top10 ~ Populism + prior_president
                              OLS Regression Results                              
Dep. Variable:     victimism_me_sim_top10   R-squared:                       0.006
Model:                                OLS   Adj. R-squared:                  0.006
Method:                     Least Squares   F-statistic:                     11.35
Date:                    Sat, 17 Jan 2026   Prob (F-statistic):           1.22e-05
Time:                            11:19:43   Log-Likelihood:                 5383.8
No. Observations:                    3722   AIC:                        -1.076e+04
Df Residuals:                        3719   BIC:                        -1.074e+04
Df Model:                               2                                         
Covariance Type:   

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '



[2] Seed loop (N=100) Populism coef:
    mean = 0.013672233207246743
    std  = 0.0012652984126980318
    min  = 0.010347001558299387
    max  = 0.016469020657708364

[3] OLS (HC3): scapegoating_them_sim_top10 ~ Populism + prior_president
                                 OLS Regression Results                                
Dep. Variable:     scapegoating_them_sim_top10   R-squared:                       0.017
Model:                                     OLS   Adj. R-squared:                  0.016
Method:                          Least Squares   F-statistic:                     31.67
Date:                         Sat, 17 Jan 2026   Prob (F-statistic):           2.30e-14
Time:                                 11:19:44   Log-Likelihood:                 5490.8
No. Observations:                         3722   AIC:                        -1.098e+04
Df Residuals:                             3719   BIC:                        -1.096e+04
Df Model:                                    2          

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '
